In [1]:
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import calibration_curve
from catboost import CatBoostClassifier, Pool
import joblib

# -------------------------------------------------------------------------
# 1. Configuration & Setup
# -------------------------------------------------------------------------
RANDOM_STATE = 42
TARGET = "Diabetes_binary"
N_SPLITS = 5

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

OUTPUT_DIR = Path("./catboost_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# 2. Data Loading & Multi-Dataset Merging
# -------------------------------------------------------------------------
PATH_DATASET1 = Path("/kaggle/input/datasets/koayzixiang/diabetes-indicator-dataset/combined_diabetes_health_indicators.csv")
PATH_DATASET2 = Path("/kaggle/input/datasets/koayzixiang/diabretes/diabetes_prediction_dataset.csv")

loaded_dfs = []

# Load Dataset 1 (BRFSS Health Indicators)
if PATH_DATASET1.exists():
    df1 = pd.read_csv(PATH_DATASET1)
    df1.columns = df1.columns.astype(str).str.strip()
    if "source_dataset" not in df1.columns:
        df1["source_dataset"] = "BRFSS_Indicators"
    loaded_dfs.append(df1)
    print(f"Loaded Dataset 1: {df1.shape}")

# Load Dataset 2 (Diabetes Prediction Dataset)
if PATH_DATASET2.exists():
    df2 = pd.read_csv(PATH_DATASET2)
    df2.columns = df2.columns.astype(str).str.strip()
    
    # Harmonize column names to align with Dataset 1
    rename_mapping = {
        "diabetes": "Diabetes_binary",
        "age": "Age",
        "gender": "Sex",
        "bmi": "BMI",
        "hypertension": "HighBP",
        "heart_disease": "HeartDiseaseorAttack",
        "smoking_history": "Smoker"
    }
    df2 = df2.rename(columns=rename_mapping)
    
    # Normalize 'Sex' encoding (Female -> 0, Male -> 1)
    if "Sex" in df2.columns and df2["Sex"].dtype == "object":
        df2["Sex"] = df2["Sex"].map({"Female": 0, "Male": 1}).fillna(df2["Sex"])

    # Convert non-numeric smoking history to a categorizable feature or indicator
    if "Smoker" in df2.columns and df2["Smoker"].dtype == "object":
        df2["Smoker"] = df2["Smoker"].map(
            {"never": 0, "no info": np.nan, "current": 1, "former": 1, "ever": 1, "not current": 0}
        )
        
    if "source_dataset" not in df2.columns:
        df2["source_dataset"] = "Diabetes_Prediction"
        
    loaded_dfs.append(df2)
    print(f"Loaded Dataset 2: {df2.shape}")

if not loaded_dfs:
    raise FileNotFoundError("No input datasets found at the specified paths.")

# Concatenate all available datasets
raw = pd.concat(loaded_dfs, axis=0, ignore_index=True)

# Retain ALL records with valid binary targets
data = raw.copy()
data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
data = data.loc[data[TARGET].isin([0, 1])].copy()
data[TARGET] = data[TARGET].astype("int8")

# Exclude multi-class target if present, keep source_dataset as feature
excluded_cols = {"Diabetes_binary", "Diabetes_012"}
feature_columns = [c for c in data.columns if c not in excluded_cols]

# Cast source_dataset as categorical string
data["source_dataset"] = data["source_dataset"].fillna("unknown").astype(str)

# Cast numerical health columns
numeric_cols = [c for c in feature_columns if c != "source_dataset"]
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Deduplicate across common numerical health metrics + target to prevent cross-dataset leakage
dedup_keys = [c for c in numeric_cols if c in data.columns and data[c].notna().sum() > 0] + [TARGET]
data = data.drop_duplicates(subset=dedup_keys).reset_index(drop=True)

# -------------------------------------------------------------------------
# 3. Domain-Specific Feature Engineering
# -------------------------------------------------------------------------
def create_health_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates metabolic interaction terms and comorbidity risk indices."""
    df = df.copy()
    
    comorbidity_cols = [c for c in ["HighBP", "HighChol", "Smoker", "Stroke", "HeartDiseaseorAttack"] if c in df.columns]
    if comorbidity_cols:
        df["Comorbidity_Score"] = df[comorbidity_cols].sum(axis=1, skipna=True)
    
    if "HighBP" in df.columns and "HighChol" in df.columns:
        df["Vascular_Risk_Interaction"] = df["HighBP"] * df["HighChol"]
        
    if "BMI" in df.columns and "Age" in df.columns:
        df["BMI_Age_Product"] = df["BMI"] * df["Age"]
        
    if "PhysHlth" in df.columns and "MentHlth" in df.columns:
        df["Total_Unhealthy_Days"] = df["PhysHlth"] + df["MentHlth"]
        
    return df

X_engineered = create_health_features(data[feature_columns])
y = data[TARGET].astype("int8").values
updated_features = list(X_engineered.columns)

print(f"\nCombined Dataset shape: {X_engineered.shape} | Positive target ratio: {y.mean():.4f}")
print("Included source datasets:", X_engineered["source_dataset"].value_counts().to_dict())

# Identify discrete categorical variables (including source_dataset)
categorical_features = ["source_dataset"]
for c in updated_features:
    if c != "source_dataset":
        # Convert integer-like low cardinality columns to integers, keep NaNs for CatBoost
        if X_engineered[c].nunique(dropna=True) <= 15 and X_engineered[c].dtype != 'float64':
            X_engineered[c] = X_engineered[c].astype("Int64")
            categorical_features.append(c)

# Convert categorical columns to string for CatBoost Pool compatibility
for cat_col in categorical_features:
    X_engineered[cat_col] = X_engineered[cat_col].astype(str).replace("<NA>", "missing").replace("nan", "missing")

# -------------------------------------------------------------------------
# 4. Stratified K-Fold Cross-Validation & Model Training
# -------------------------------------------------------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X_engineered))
models = []

catboost_params = {
    "iterations": 2500,
    "depth": 7,
    "learning_rate": 0.035,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "auto_class_weights": "Balanced",
    "verbose": 0,
    "allow_writing_files": False
}

print(f"\n--- Starting {N_SPLITS}-Fold Stratified Cross-Validation ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_engineered, y), start=1):
    X_train, y_train = X_engineered.iloc[train_idx], y[train_idx]
    X_val, y_val = X_engineered.iloc[val_idx], y[val_idx]
    
    train_pool = Pool(X_train, y_train, cat_features=categorical_features)
    val_pool = Pool(X_val, y_val, cat_features=categorical_features)
    
    model = CatBoostClassifier(**catboost_params)
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=120,
        use_best_model=True
    )
    
    val_probs = model.predict_proba(val_pool)[:, 1]
    oof_preds[val_idx] = val_probs
    models.append(model)
    
    fold_auc = roc_auc_score(y_val, val_probs)
    print(f"Fold {fold} Best Iteration: {model.get_best_iteration()} | Validation ROC-AUC: {fold_auc:.5f}")

oof_auc = roc_auc_score(y, oof_preds)
oof_pr_auc = average_precision_score(y, oof_preds)
print(f"\nOverall Out-of-Fold ROC-AUC: {oof_auc:.5f} | PR-AUC: {oof_pr_auc:.5f}")

# -------------------------------------------------------------------------
# 5. Threshold Optimization
# -------------------------------------------------------------------------
thresholds = np.arange(0.10, 0.91, 0.025)
threshold_metrics = []

for t in thresholds:
    preds = (oof_preds >= t).astype(int)
    threshold_metrics.append({
        "threshold": round(float(t), 3),
        "f1": f1_score(y, preds, zero_division=0),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0)
    })

thresh_df = pd.DataFrame(threshold_metrics)
best_idx = thresh_df["f1"].idxmax()
optimal_threshold = float(thresh_df.loc[best_idx, "threshold"])

print(f"\nOptimal Decision Threshold (Max F1): {optimal_threshold:.3f}")
print(f"OOF F1-Score: {thresh_df.loc[best_idx, 'f1']:.4f} | Precision: {thresh_df.loc[best_idx, 'precision']:.4f} | Recall: {thresh_df.loc[best_idx, 'recall']:.4f}")

# -------------------------------------------------------------------------
# 6. Evaluation & Visualizations
# -------------------------------------------------------------------------
oof_binary_preds = (oof_preds >= optimal_threshold).astype(int)

print("\n--- Out-Of-Fold Classification Report ---")
print(classification_report(y, oof_binary_preds, target_names=["No Diabetes", "Diabetes"], zero_division=0))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

cm = confusion_matrix(y, oof_binary_preds, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0, 0],
            xticklabels=["Predicted No", "Predicted Yes"],
            yticklabels=["Actual No", "Actual Yes"])
axes[0, 0].set_title(f"OOF Confusion Matrix (Threshold = {optimal_threshold:.3f})")

RocCurveDisplay.from_predictions(y, oof_preds, ax=axes[0, 1], name="CatBoost Combined OOF")
axes[0, 1].set_title("Out-Of-Fold ROC Curve")

PrecisionRecallDisplay.from_predictions(y, oof_preds, ax=axes[1, 0], name="CatBoost Combined OOF")
axes[1, 0].set_title("Out-Of-Fold Precision-Recall Curve")

prob_true, prob_pred = calibration_curve(y, oof_preds, n_bins=10, strategy="quantile")
axes[1, 1].plot(prob_pred, prob_true, marker="o", label="CatBoost Combined OOF")
axes[1, 1].plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
axes[1, 1].set_xlabel("Mean Predicted Probability")
axes[1, 1].set_ylabel("Observed Fraction of Positives")
axes[1, 1].set_title("Probability Calibration Curve")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "oof_combined_evaluation_dashboard.png", dpi=300)
plt.show()

# -------------------------------------------------------------------------
# 7. Final Model Retraining & Artifact Saving
# -------------------------------------------------------------------------
print("\n--- Retraining Final Production Model on Combined Dataset ---")

final_pool = Pool(X_engineered, y, cat_features=categorical_features)
final_model = CatBoostClassifier(**catboost_params)

avg_best_iteration = int(np.mean([m.get_best_iteration() for m in models]))
final_model.set_params(iterations=max(avg_best_iteration, 100))
final_model.fit(final_pool, verbose=200)

final_model_path = OUTPUT_DIR / "catboost_combined_diabetes_model.cbm"
final_model.save_model(str(final_model_path))

pipeline_metadata = {
    "feature_names": updated_features,
    "categorical_features": categorical_features,
    "optimal_threshold": optimal_threshold,
    "oof_auc": oof_auc,
    "oof_pr_auc": oof_pr_auc
}
joblib.dump(pipeline_metadata, OUTPUT_DIR / "combined_model_metadata.joblib")

print(f"\nFinal combined model saved to: {final_model_path}")

FileNotFoundError: No input datasets found at the specified paths.